# 🗄️ BDCN 3A EEBFM — Semana 10 + Supabase + DT
**Professor:** Heglas Oliveira | heglasoliveira@professor.educacao.sp.gov.br  
**Escola:** EEBFM Jacareí-SP  
**Disciplina:** Banco de Dados e Computação em Nuvem (BDCN)  
**Turma:** 3A | **Data:** 13/05/2026 | Multiplica SP 2026.1  

**Objetivo:** Criar banco relacional com dados reais da turma, aplicar Design Thinking, executar consultas SQL e publicar em nuvem gratuita (Supabase/Neon).  
**Canvas DT oficial:** [Design Thinking Canvas no Figma](https://www.figma.com/make/k8nGKP4fR2ryHKgoN5Bwae/Design-Thinking-Canvas?fullscreen=1&t=2wluPP7VBojC6fGR-1&code-node-id=0-9)


## 🎨 Canvas DT oficial
A referência visual da atividade está neste link: [Design Thinking Canvas no Figma](https://www.figma.com/make/k8nGKP4fR2ryHKgoN5Bwae/Design-Thinking-Canvas?fullscreen=1&t=2wluPP7VBojC6fGR-1&code-node-id=0-9).
Use esse canvas para conduzir as etapas de Empatia, Definição, Ideação, Prototipagem e Teste.


In [ ]:
!pip install pandas matplotlib seaborn -q
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')
print('✅ Bibliotecas prontas!')

## 🧭 Roteiro da aula
1. Acolhida e apresentação do problema.
2. Aplicação do Canvas DT.
3. Criação das tabelas e inserção de dados.
4. Execução das consultas SQL.
5. Publicação em Supabase.
6. Registro de evidências para o portfólio.


In [ ]:
conn = sqlite3.connect('bdcn_3a.db')
c = conn.cursor()
c.execute('''CREATE TABLE IF NOT EXISTS alunos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    ra TEXT UNIQUE,
    situacao TEXT
)''')
c.execute('''CREATE TABLE IF NOT EXISTS notas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    id_aluno INTEGER,
    disciplina TEXT,
    media REAL CHECK(media >= 0 AND media <= 10),
    FOREIGN KEY (id_aluno) REFERENCES alunos(id)
)''')
conn.commit()
print('✅ Tabelas alunos e notas criadas!')

In [ ]:
alunos_data = [
    ('Ana Beatriz dos Santos Carvalho', '00001127258357sp', 'B1'),
    ('Ana Carolina Lemes Barbarossi',   '00001126988443sp', 'B1'),
    ('Ana Julia da Silva',              '00001089973597sp', 'B1'),
    ('Cassio Nunes Oliveira Santana',   '0000112221490xsp', 'Ativo'),
    ('Julia Silva Nogueira Medeiros',   '00001105525661sp', 'Ativo'),
    ('Miguel de Paula Moura',           '00001109239117sp', 'Ativo'),
    ('Nicole dos Santos Santana',       '00001122893127sp', 'Ativo'),
    ('Otavio Henrique Nunes',           '00001134526829sp', 'L1'),
    ('Rayane Rylari Bispo da Silva',    '00001115930345sp', 'B2'),
    ('Rayssa Pamela Lopes Rodrigues',   '00001131380861sp', 'Ativo'),
]
notas_data = [
    (1, 'BDCN', 3.0),
    (2, 'BDCN', 6.0),
    (3, 'BDCN', 9.0),
    (4, 'BDCN', 8.0),
    (5, 'BDCN', 7.0),
    (6, 'BDCN', 8.0),
    (7, 'BDCN', 9.0),
    (8, 'BDCN', 7.0),
    (9, 'BDCN', 9.0),
    (10,'BDCN', 9.0),
]
c.executemany('INSERT OR IGNORE INTO alunos (nome, ra, situacao) VALUES (?,?,?)', alunos_data)
c.executemany('INSERT OR IGNORE INTO notas (id_aluno, disciplina, media) VALUES (?,?,?)', notas_data)
conn.commit()
print('✅ Dados turma 3A inseridos com sucesso!')

## 📋 Consultas SQL
Execute as células abaixo e tire print dos resultados para o portfólio Multiplica SP.

In [ ]:
# CONSULTA 1 - Alunos em recuperacao
df_rec = pd.read_sql_query(
    """SELECT a.nome, a.ra, a.situacao, n.media
       FROM alunos a
       JOIN notas n ON a.id = n.id_aluno
       WHERE n.media < 5 AND n.disciplina = 'BDCN'
       ORDER BY n.media""", conn)
print('=== Alunos em Recuperacao BDCN ===')
print(df_rec)

In [ ]:
# CONSULTA 2 - Top 5 desempenho
df_top = pd.read_sql_query(
    """SELECT a.nome, n.media
       FROM alunos a
       JOIN notas n ON a.id = n.id_aluno
       WHERE n.disciplina = 'BDCN'
       ORDER BY n.media DESC LIMIT 5""", conn)
print('=== Top 5 Desempenho BDCN ===')
print(df_top)

In [ ]:
# CONSULTA 3 - Media por situacao
df_media = pd.read_sql_query(
    """SELECT a.situacao,
              ROUND(AVG(n.media),2) AS media_situacao,
              COUNT(*) AS qtd
       FROM alunos a
       JOIN notas n ON a.id = n.id_aluno
       GROUP BY a.situacao
       ORDER BY media_situacao DESC""", conn)
print('=== Media por Situacao ===')
print(df_media)

## 📊 Visualização
Gráficos gerados a partir das notas do banco.

In [ ]:
# VISUALIZACAO - Graficos
df_all = pd.read_sql_query('SELECT * FROM notas', conn)
df_alunos = pd.read_sql_query('SELECT * FROM alunos', conn)
df_join = df_all.merge(df_alunos, left_on='id_aluno', right_on='id')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(df_join['nome'].str.split().str[0],
            df_join['media'], color='skyblue', edgecolor='navy')
axes[0].set_title('Medias BDCN - Turma 3A EEBFM')
axes[0].set_xlabel('Aluno')
axes[0].set_ylabel('Media')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(y=5, color='red', linestyle='--', label='Minimo 5')
axes[0].legend()

axes[1].hist(df_join['media'], bins=5,
             color='lightgreen', edgecolor='darkgreen')
axes[1].set_title('Distribuicao das Notas BDCN')
axes[1].set_xlabel('Media')
axes[1].set_ylabel('Frequencia')
axes[1].axvline(x=5, color='red', linestyle='--', label='Minimo 5')
axes[1].legend()

plt.tight_layout()
plt.savefig('bdcn_3a_analise.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo: bdcn_3a_analise.png')

## ☁️ Deploy Supabase — Passo a Passo
1. Acesse **https://supabase.com/dashboard/sign-up** (sem cartão).
2. Clique **New Project** > Nome: `bdcn_3a_seu_nome` > Região: South America > Create.
3. Menu esquerdo > **SQL Editor** > **New query**.
4. Cole o SQL da próxima célula > clique **RUN**.
5. Veja a tabela criada e o resultado da consulta.
6. **Screenshot** para o portfólio Multiplica SP! 📸

**Alternativa:** Neon PostgreSQL free em **https://console.neon.tech** (mesmo SQL).

In [ ]:
sql_supabase = """
-- ================================================
-- COPIE ESTE BLOCO NO SUPABASE > SQL EDITOR > RUN
-- ================================================

CREATE TABLE alunos (
    id SERIAL PRIMARY KEY,
    nome TEXT NOT NULL,
    ra TEXT UNIQUE,
    situacao TEXT
);

INSERT INTO alunos (nome, ra, situacao) VALUES
('Ana Beatriz dos Santos Carvalho', '00001127258357sp', 'B1'),
('Ana Carolina Lemes Barbarossi',   '00001126988443sp', 'B1'),
('Ana Julia da Silva',              '00001089973597sp', 'B1'),
('Cassio Nunes Oliveira Santana',   '0000112221490xsp', 'Ativo'),
('Julia Silva Nogueira Medeiros',   '00001105525661sp', 'Ativo'),
('Miguel de Paula Moura',           '00001109239117sp', 'Ativo'),
('Nicole dos Santos Santana',       '00001122893127sp', 'Ativo'),
('Otavio Henrique Nunes',           '00001134526829sp', 'L1'),
('Rayane Rylari Bispo da Silva',    '00001115930345sp', 'B2'),
('Rayssa Pamela Lopes Rodrigues',   '00001131380861sp', 'Ativo');

CREATE TABLE notas (
    id SERIAL PRIMARY KEY,
    id_aluno INTEGER REFERENCES alunos(id),
    disciplina TEXT,
    media DECIMAL CHECK(media >= 0 AND media <= 10)
);

INSERT INTO notas (id_aluno, disciplina, media) VALUES
(1, 'BDCN', 3.0), (2, 'BDCN', 6.0), (3, 'BDCN', 9.0),
(4, 'BDCN', 8.0), (5, 'BDCN', 7.0), (6, 'BDCN', 8.0),
(7, 'BDCN', 9.0), (8, 'BDCN', 7.0), (9, 'BDCN', 9.0),
(10,'BDCN', 9.0);

-- CONSULTA 1: Alunos em recuperacao
SELECT a.nome, n.media
FROM alunos a JOIN notas n ON a.id = n.id_aluno
WHERE n.media < 5;

-- CONSULTA 2: Top 5
SELECT a.nome, n.media
FROM alunos a JOIN notas n ON a.id = n.id_aluno
ORDER BY n.media DESC LIMIT 5;

-- CONSULTA 3: Media por situacao
SELECT a.situacao, ROUND(AVG(n.media),2) AS media_situacao, COUNT(*) AS qtd
FROM alunos a JOIN notas n ON a.id = n.id_aluno
GROUP BY a.situacao
ORDER BY media_situacao DESC;
"""
print(sql_supabase)

## 📸 Evidências esperadas
Ao final da atividade, o portfólio pode conter:
- Print do Canvas DT preenchido no Figma.
- Print das tabelas criadas no Colab.
- Print das consultas SQL executadas.
- Print da publicação no Supabase ou Neon.
- Depoimentos curtos dos alunos.
- Foto da turma trabalhando em grupo.
- Captura do gráfico gerado em Python.


## ✅ Finalização
Banco fechado. Execute a célula abaixo e confira os próximos passos para Supabase.

In [ ]:
conn.close()
print('Banco local fechado.')
print()
print('PROXIMOS PASSOS SUPABASE:')
print('1. Acesse: https://supabase.com/dashboard/sign-up')
print('2. Crie projeto: bdcn_3a_seu_nome | Regiao: South America')
print('3. Va em SQL Editor > Cole o SQL da celula anterior > Run')
print('4. Screenshot do resultado para o portfolio Multiplica SP!')
print()
print('EVIDENCIAS PARA O PORTFOLIO:')
print('- Canvas DT preenchido')
print('- Prints das consultas SQL')
print('- Screenshot do Supabase')
print('- Depoimentos dos alunos')
print()
print('ALTERNATIVA NEON (1GB free): https://console.neon.tech')
print()
print('Arquivo para download: BDCN_3A_Supabase.ipynb')
print('Atividade concluida! Bons estudos, turma 3A!')
